# Classificação supervisionada com SMILE RANDOM FOREST

Este roteiro usa os polígonos salvos como assets no Earth Engine para treinar e validar uma classificação binária:

- `landcover = 0`: floresta;
- `landcover = 1`: agricultura.

Fluxo: inicializar → criar o feature space → carregar e rotular os polígonos → separar polígonos inteiros de treino/validação → amostrar cada conjunto → treinar CART → classificar → avaliar.

In [1]:
import ee

from repository_ee import (
    GeometryUtils,
    Landsat8,
    LocalLabels,
    MapLayers,
    Models,
    PeriodoAgricola,
    initialize_earth_engine,
)

## 1. Inicialização e parâmetros gerais

A AOI e o ano precisam ser os mesmos usados na criação das amostras.

In [2]:
initialize_earth_engine(use_eerepr=True)

ANO = 2025
ESCALA = 30
SEED = 42
FRACAO_TREINO = 0.70
AMOSTRAS_POR_CLASSE = 1000
PROPRIEDADE_CLASSE = "landcover"

COORDS_TOCANTINS = [
    [-48.46995289774661, -10.859497023647087],
    [-47.84922535868411, -10.859497023647087],
    [-47.84922535868411, -10.362094203375317],
    [-48.46995289774661, -10.362094203375317],
    [-48.46995289774661, -10.859497023647087],
]

aoi = GeometryUtils.polygon(COORDS_TOCANTINS)
periodo = PeriodoAgricola.tocantins(ANO)

/home/samuel/.venvs/jupyter/lib/python3.12/site-packages/ee/data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(


## 2. Imagem e variáveis preditoras

O `feature_space` reúne as bandas Landsat e as métricas temporais calculadas pelo repositório. A mesma lista e ordem de bandas será usada no treinamento e na classificação.

In [ ]:
landsat = Landsat8.collection(aoi, periodo)
feature_space = Landsat8.feature_space(landsat, aoi, periodo)
bandas_preditoras = feature_space.bandNames()

print("Quantidade de bandas preditoras:", bandas_preditoras.size().getInfo())
print("Bandas:", bandas_preditoras.getInfo())

## 3. Carregamento e rotulagem dos polígonos

O campo `landcover` é definido novamente com base no asset de origem. Isso é intencional: os dois arquivos podem conter o mesmo `class_id` usado durante o desenho.

In [ ]:
ASSET_FLORESTA = (
    "projects/poetic-tesla-506814-e1/assets/tocantins_2025_FLORESTA"
)
ASSET_AGRICULTURA = (
    "projects/poetic-tesla-506814-e1/assets/tocantins_2025_AGRICULTURA"
)

ASSET_AGUA = (
    "projects/poetic-tesla-506814-e1/assets/tocantins_2025_AGUA"
)




poligonos_floresta = ee.FeatureCollection(ASSET_FLORESTA).map(
    lambda feature: feature.set(PROPRIEDADE_CLASSE, 0).set("classe_nome", "FLORESTA")
).filterBounds(aoi)
poligonos_agricultura = ee.FeatureCollection(ASSET_AGRICULTURA).map(
    lambda feature: feature.set(PROPRIEDADE_CLASSE, 1).set("classe_nome", "AGRICULTURA")
).filterBounds(aoi)
poligonos_agua = ee.FeatureCollection(ASSET_AGUA).map(
    lambda feature: feature.set(PROPRIEDADE_CLASSE, 2).set("classe_nome", "AGUA")
).filterBounds(aoi)


poligonos = poligonos_floresta.merge(poligonos_agricultura).merge(poligonos_agua)

print("Polígonos de floresta:", poligonos_floresta.size().getInfo())
print("Polígonos de agricultura:", poligonos_agricultura.size().getInfo())
print("Polígonos de água:", poligonos_agua.size().getInfo())
print("Distribuição dos polígonos:", Models.class_distribution(
    poligonos, PROPRIEDADE_CLASSE
).getInfo())

## 4. Separação dos polígonos de treinamento e validação

A divisão é feita antes da amostragem e separadamente por classe. Assim, cada polígono inteiro pertence somente ao treinamento ou à validação, e ambas as classes podem ser verificadas nos dois conjuntos.

In [ ]:
floresta_com_split = poligonos_floresta.randomColumn(
    columnName="split",
    seed=SEED,
)
agricultura_com_split = poligonos_agricultura.randomColumn(
    columnName="split",
    seed=SEED + 1,
)
agua_com_split = poligonos_agua.randomColumn(
    columnName="split",
    seed=SEED + 2,
)

poligonos_treino = (
    floresta_com_split.filter(ee.Filter.lt("split", FRACAO_TREINO))
    .merge(agricultura_com_split.filter(ee.Filter.lt("split", FRACAO_TREINO)))
    .merge(agua_com_split.filter(ee.Filter.lt("split", FRACAO_TREINO)))
    
)
poligonos_validacao = (
    floresta_com_split.filter(ee.Filter.gte("split", FRACAO_TREINO))
    .merge(agricultura_com_split.filter(ee.Filter.gte("split", FRACAO_TREINO)))
    .merge(agua_com_split.filter(ee.Filter.gte("split", FRACAO_TREINO)))
    
)

distribuicao_poligonos_treino = Models.class_distribution(
    poligonos_treino, PROPRIEDADE_CLASSE
).getInfo()
distribuicao_poligonos_validacao = Models.class_distribution(
    poligonos_validacao, PROPRIEDADE_CLASSE
).getInfo()

if (
    len(distribuicao_poligonos_treino) < 2
    or len(distribuicao_poligonos_validacao) < 2
):
    raise ValueError(
        "Treino e validação precisam conter polígonos de floresta, agricultura, água e urbano. "
        "Desenhe mais polígonos ou altere o SEED."
    )

print("Polígonos de treinamento:", distribuicao_poligonos_treino)
print("Polígonos de validação:", distribuicao_poligonos_validacao)

In [ ]:
mapa_poligonos = MapLayers.create(aoi, zoom=9)
mapa_poligonos.addLayer(
    poligonos_treino, {"color": "1a9850"}, "Polígonos — treinamento"
)
mapa_poligonos.addLayer(
    poligonos_validacao, {"color": "d73027"}, "Polígonos — validação"
)
mapa_poligonos

## 5. Rasterização e amostragem independente

Cada conjunto de polígonos é convertido separadamente em uma banda `landcover`. Em seguida, `Models.stratified_sample()` extrai amostras balanceadas de treino e validação sem reutilizar um mesmo polígono nos dois conjuntos.

In [ ]:
labels_treino = LocalLabels.from_feature_collection(
    features=poligonos_treino,
    class_property=PROPRIEDADE_CLASSE,
    geometry=aoi,
    reference_image=feature_space,
    band_name=PROPRIEDADE_CLASSE,
)
labels_validacao = LocalLabels.from_feature_collection(
    features=poligonos_validacao,
    class_property=PROPRIEDADE_CLASSE,
    geometry=aoi,
    reference_image=feature_space,
    band_name=PROPRIEDADE_CLASSE,
)

amostras_treino = Models.stratified_sample(
    feature_space=feature_space,
    labels=labels_treino,
    class_band=PROPRIEDADE_CLASSE,
    region=aoi,
    num_points=AMOSTRAS_POR_CLASSE,
    class_values=[0, 1, 2],
    class_points=[AMOSTRAS_POR_CLASSE, AMOSTRAS_POR_CLASSE],
    scale=ESCALA,
    seed=SEED,
    tile_scale=2,
    geometries=True,
)
amostras_validacao = Models.stratified_sample(
    feature_space=feature_space,
    labels=labels_validacao,
    class_band=PROPRIEDADE_CLASSE,
    region=aoi,
    num_points=AMOSTRAS_POR_CLASSE,
    class_values=[0, 1, 2],
    class_points=[AMOSTRAS_POR_CLASSE, AMOSTRAS_POR_CLASSE],
    scale=ESCALA,
    seed=SEED + 1,
    tile_scale=2,
    geometries=True,
)

distribuicao_treino = Models.class_distribution(
    amostras_treino, PROPRIEDADE_CLASSE
).getInfo()
distribuicao_validacao = Models.class_distribution(
    amostras_validacao, PROPRIEDADE_CLASSE
).getInfo()

if len(distribuicao_treino) < 3 or len(distribuicao_validacao) < 3:
    raise ValueError(
        "Treino e validação precisam conter floresta, agricultura, água  "
        "Desenhe mais polígonos ou altere o SEED."
    )

print("Amostras de treinamento:", distribuicao_treino)
print("Amostras de validação:", distribuicao_validacao)

## 6. Treinamento do SIMPLE RANDOM FOREST

A criação e o treinamento do classificador passam pela classe `Models`. `max_nodes` controla o tamanho máximo da árvore e `min_leaf_population` evita folhas sustentadas por pouquíssimos pixels.

In [ ]:
modelo_rf = Models.train(
    samples=amostras_treino,
    input_properties=bandas_preditoras,
    class_property=PROPRIEDADE_CLASSE,
    algorithm="rf",
    classifier_parameters={
        "number_of_trees": 100,
        "variables_per_split": 5,
        "bag_fraction": 0.8,
        "max_nodes": 100,
        "min_leaf_population": 10,
        
    },
)

modelo_rf.explain()

## 7. Classificação da imagem

A imagem é classificada com exatamente as mesmas bandas, na mesma ordem, usadas pelo treinamento.

In [ ]:
classificacao_rf = Models.classify_image(
    feature_space=feature_space,
    classifier=modelo_rf,
    input_properties=bandas_preditoras,
    output_band=PROPRIEDADE_CLASSE,
).clip(aoi)

VIS_RF = {
    "bands": [PROPRIEDADE_CLASSE],
    "min": 0,
    "max": 3,
    "palette": ["006400", "ffcc00","0000ff","ff0000"],
}

mapa = MapLayers.create(aoi, zoom=9)
MapLayers.add_rgb(mapa, feature_space, "Landsat — RGB")
mapa.addLayer(classificacao_rf, VIS_RF, "RF — floresta x agricultura")
mapa.addLayer(poligonos_floresta, {"color": "006400"}, "Polígonos — floresta")
mapa.addLayer(poligonos_agricultura, {"color": "ffcc00"}, "Polígonos — agricultura")
mapa.addLayer(poligonos_agua, {"color": "0000ff"}, "Polígonos — água")
mapa.addLayer(poligonos_urban, {"color": "ff0000"}, "Polígonos — urbano")
mapa

## 8. Avaliação do treinamento e validação independente

A matriz de treinamento mede o ajuste aos dados vistos pelo CART. A matriz de validação é a referência principal, pois usa pixels extraídos exclusivamente dos polígonos reservados para validação.

In [ ]:
metricas_treino = Models.training_metrics(modelo_rf)
metricas_validacao = Models.validation_metrics(
    classifier=modelo_rf,
    validation_samples=amostras_validacao,
    class_property=PROPRIEDADE_CLASSE,
    prediction_property="classification",
    class_order=[0, 1, 2, 3],
)
print("#### RANDOM FOREST ACURACY ####")
print("Matriz de confusão — treinamento:")
print(metricas_treino["confusion_matrix"].getInfo())
print("Acurácia de treinamento:", metricas_treino["accuracy"].getInfo())

print("\nMatriz de confusão — validação:")
print(metricas_validacao["confusion_matrix"].getInfo())
print("Acurácia de validação:", metricas_validacao["accuracy"].getInfo())
print("Kappa de validação:", metricas_validacao["kappa"].getInfo())
print("Acurácia do produtor:", metricas_validacao["producers_accuracy"].getInfo())
print("Acurácia do consumidor:", metricas_validacao["consumers_accuracy"].getInfo())

## Observação metodológica

Este roteiro separa polígonos inteiros antes da extração dos pixels, evitando que o mesmo polígono forneça amostras simultaneamente ao treinamento e à validação. Polígonos distintos ainda podem ser vizinhos; quando necessário, uma faixa de exclusão espacial pode complementar esta divisão.